## Lab 2 — OOP Refactor & FastAPI CRUD API


## 0 — Setup

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'fastapi', 'uvicorn[standard]', 'httpx', 'pydantic', '-q'])
print('✅ Dependencies installed')

✅ Dependencies installed


---
## Task 1 — Model the Server with a Dataclass

### 📖 Concept: `@dataclass`

A `dataclass` auto-generates `__init__`, `__repr__`, and `__eq__` from field annotations. Use it for data containers that don't need complex logic.

```python
from dataclasses import dataclass, field

@dataclass
class Config:
    host: str
    port: int = 8080
    tags: list[str] = field(default_factory=list)  # mutable defaults need field()

cfg = Config('localhost')
print(cfg)  # Config(host='localhost', port=8080, tags=[])
```

> Never use `tags: list = []` as a default — that creates one shared list for all instances. Always use `field(default_factory=list)`.

In [2]:
from dataclasses import dataclass, field

# ✏️ YOUR TURN
# Create a Server dataclass with fields:
#   id: int
#   name: str
#   host: str
#   port: int
#   status: str = 'unknown'
#   tags: list[str] = (use field(default_factory=list))
#
# Add a method base_url(self) -> str that returns 'http://{host}:{port}'

@dataclass
class Server:
    id: int
    name: str
    host: str
    port: int
    status: str = "unknown"
    tags: list[str] = field(default_factory=list)

    def base_url(self) -> str:
        return f"http://{self.host}:{self.port}"
s = Server(id=1, name='api-prod-1', host='10.0.0.1', port=8080)
print(s)
print(s.base_url())

Server(id=1, name='api-prod-1', host='10.0.0.1', port=8080, status='unknown', tags=[])
http://10.0.0.1:8080


---
## Task 2 — ConfigLoader Class

### 📖 Concept: Classes with logging

A class groups related data (the file path) and behaviour (the `load()` method). Use `logging` instead of `print()` for anything that goes to production.

```python
import logging
logger = logging.getLogger(__name__)
logger.info('Loading %d items', count)   # use %s placeholders, not f-strings
```

In [3]:
import json
import logging
import pathlib

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)


class ConfigError(ValueError):
    """Raised when config loading fails."""
    pass


# ✏️ YOUR TURN
# Implement ConfigLoader:
#   __init__(self, path: str) — store path as pathlib.Path
#   load(self) -> list[Server] — parse JSON, return list of Server objects
#   Raise ConfigError on FileNotFoundError or JSONDecodeError
#   Log info on success

class ConfigLoader:
    """Loads server configuration from a JSON file."""

    def __init__(self, path: str):
        self.path = pathlib.Path(path)

    def load(self) -> list:
        """Parse the config file and return Server objects."""
        logger.info("Loading config from %s", self.path)
        try:
            raw = json.loads(self.path.read_text())
        except FileNotFoundError:
            logger.error("Config file not found: %s", self.path)
            raise ConfigError(f"File not found: {self.path}")
        except json.JSONDecodeError as e:
            logger.error("Invalid JSON: %s", e)
            raise ConfigError(f"Invalid JSON: {e}") from e

        servers = []
        for i, entry in enumerate(raw, start=1):
            servers.append(Server(
                id=i,
                name=entry["name"],
                host=entry["host"],
                port=entry["port"],
            ))
        logger.info("Loaded %d servers", len(servers))
        return servers


# Test it
loader = ConfigLoader('servers.json')
server_objects = loader.load()
print(server_objects)

2026-06-30 12:09:34,843 [INFO] Loading config from servers.json
2026-06-30 12:09:34,849 [INFO] Loaded 3 servers


[Server(id=1, name='api-prod-1', host='httpbin.org', port=443, status='unknown', tags=[]), Server(id=2, name='api-prod-2', host='httpbin.org', port=443, status='unknown', tags=[]), Server(id=3, name='unreachable', host='10.255.255.1', port=8080, status='unknown', tags=[])]


---
## Task 3 — HealthChecker Class (async)

### 📖 Concept: `async`/`await` and `asyncio.gather()`

An `async def` function is a coroutine — it can pause and yield control while waiting for I/O (network, disk). This lets Python handle other work in the meantime.

`asyncio.gather()` runs multiple coroutines **concurrently** — instead of checking 10 servers one by one (10 × 1s = 10s), it checks all 10 at once (~1s total).

```python
import asyncio, httpx

async def fetch(url: str) -> int:
    async with httpx.AsyncClient() as client:
        resp = await client.get(url)
        return resp.status_code

# Run one coroutine
code = asyncio.run(fetch('https://example.com'))

# Run many concurrently
codes = asyncio.run(asyncio.gather(fetch('https://a.com'), fetch('https://b.com')))
```

In [4]:
import asyncio
import time
import httpx

# ✏️ YOUR TURN
# Implement HealthChecker:
#   __init__(self, timeout=5.0, degraded_threshold_ms=500.0)
#   async check(self, server: Server) -> Server
#     — GET {server.base_url()}/health with httpx.AsyncClient
#     — Set server.status to UP / DEGRADED / DOWN
#     — Return the server
#   async check_all(self, servers: list[Server]) -> list[Server]
#     — Use asyncio.gather() to check all servers concurrently

class HealthChecker:
    """Checks server health over HTTP asynchronously."""

    def __init__(self, timeout: float = 5.0, degraded_threshold_ms: float = 500.0):
        self.timeout = timeout
        self.degraded_threshold_ms = degraded_threshold_ms

    async def check(self, server) -> object:
        url = f"{server.base_url()}/health"
        start = time.time()
        try:
            async with httpx.AsyncClient(timeout=self.timeout) as client:
                resp = await client.get(url)
            elapsed_ms = (time.time() - start) * 1000
            if resp.status_code == 200 and elapsed_ms <= self.degraded_threshold_ms:
                server.status = "UP"
            elif resp.status_code == 200:
                server.status = "DEGRADED"
            else:
                server.status = "DEGRADED"
        except (httpx.ConnectError, httpx.TimeoutException):
            server.status = "DOWN"
        return server

    async def check_all(self, servers: list) -> list:
        return await asyncio.gather(*[self.check(s) for s in servers])


# Quick test with a real server
checker = HealthChecker()
test_server = Server(id=99, name='httpbin', host='httpbin.org', port=443)
# We can't use asyncio.run() inside Jupyter — use await directly:
result = await checker.check(test_server)
print(result)

2026-06-30 12:09:43,022 [INFO] HTTP Request: GET http://httpbin.org:443/health "HTTP/1.1 400 Bad Request"


Server(id=99, name='httpbin', host='httpbin.org', port=443, status='DEGRADED', tags=[])


---
## Task 4 — FastAPI CRUD API

### 📖 Concept: Pydantic models for validation

FastAPI uses Pydantic models to validate incoming request bodies and serialise responses automatically.

```python
from pydantic import BaseModel, Field

class ServerIn(BaseModel):
    name: str
    port: int = Field(default=8080, ge=1, le=65535)  # validated automatically
```

FastAPI raises HTTP 422 automatically if validation fails — no manual checking needed.

---

The cells below write the full API to disk. Run the terminal command after to test it.

In [ ]:
# Task 4 — Test the FastAPI CRUD API (lab2_main.py)
import sys
sys.path.insert(0, ".")

from fastapi.testclient import TestClient
import lab2_main

# Reset in-memory store for a clean demo
lab2_main._store.clear()
lab2_main._counter = 0

client = TestClient(lab2_main.app)

# 1. API health
print("GET /health:", client.get("/health").json())

# 2. Register 3 servers
for name in ["api-prod-1", "api-prod-2", "api-prod-3"]:
    r = client.post("/servers", json={"name": name, "host": "httpbin.org", "port": 443})
    print(f"POST /servers ({name}):", r.status_code, r.json()["status"])

# 3. List all servers
print("\nGET /servers:", [s["name"] for s in client.get("/servers").json()])

# 4. Get one server
print("GET /servers/1:", client.get("/servers/1").json()["name"])

# 5. Trigger health check
checked = client.post("/servers/1/check").json()
print(f"POST /servers/1/check: status={checked['status']}")

# 6. Filter by status
print("GET /servers?status=UP:", client.get("/servers", params={"status": "UP"}).json())

# 7. Delete server 2
print("DELETE /servers/2:", client.delete("/servers/2").status_code)
print("Remaining:", [s["name"] for s in client.get("/servers").json()])

### Run the API with Swagger UI

Open a terminal in `Day2/labs/` and run:

```bash
cd Day2/labs
source ../../venv/bin/activate
uvicorn lab2_main:app --reload --port 8000
```

Then visit: **http://localhost:8000/docs**